In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lower, regexp_replace, substring, length

spark = (
    SparkSession.builder
    .appName("vulnerability-clustering")
    .config("spark.driver.memory", "4g")
    .config("spark.executor.memory", "4g")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/25 10:32:03 WARN Utils: Your hostname, CrisBook.local, resolves to a loopback address: 127.0.0.1; using 192.168.13.159 instead (on interface en0)
26/04/25 10:32:03 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/25 10:32:04 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
df = spark.read.parquet("../data/dev/gold/vulnerability_scores")

df_text = df.select(
    "cve_id",
    "description",
    "cwe",
    "cvss_severity",
    "priority_score"
).dropna(subset=["description"])

In [3]:
df_text = df_text.withColumn(
    "description_short",
    substring(col("description"), 1, 1000)
)

In [4]:
df_text = (
    df_text
    .withColumn("clean_text", lower(col("description_short")))
    .withColumn("clean_text", regexp_replace(col("clean_text"), r"\n", " "))
    .withColumn("clean_text", regexp_replace(col("clean_text"), r"http\S+", " "))
    .withColumn("clean_text", regexp_replace(col("clean_text"), r"cve-\d{4}-\d+", " "))
    .withColumn("clean_text", regexp_replace(col("clean_text"), r"0x[0-9a-f]+", " "))
    .withColumn("clean_text", regexp_replace(col("clean_text"), r"[^a-zA-Z\s]", " "))
    .withColumn("clean_text", regexp_replace(col("clean_text"), r"\s+", " "))
)

In [5]:
from pyspark.ml.feature import RegexTokenizer

tokenizer = RegexTokenizer(
    inputCol="clean_text",
    outputCol="words",
    pattern="\\W+",
    minTokenLength=3
)

df_tok = tokenizer.transform(df_text)

In [6]:
from pyspark.ml.feature import StopWordsRemover

custom_stopwords = [
    "vulnerability", "vulnerabilities", "affected", "affects",
    "issue", "issues", "allows", "attacker", "attack",
    "remote", "local", "authenticated", "unauthenticated",
    "successful", "exploitation", "exploit", "exploited",
    "crafted", "malicious", "user", "users",
    "version", "versions", "prior", "before",
    "following", "resolved", "kernel", "linux",
    "trace", "call", "task", "error", "warning",
    "function", "file", "system", "data",
    "may", "could", "can"
]

default_stopwords = StopWordsRemover.loadDefaultStopWords("english")
all_stopwords = list(set(default_stopwords + custom_stopwords))

remover = StopWordsRemover(
    inputCol="words",
    outputCol="filtered_words",
    stopWords=all_stopwords
)

df_clean = remover.transform(df_tok)

In [7]:
from pyspark.ml.feature import CountVectorizer, IDF

vectorizer = CountVectorizer(
    inputCol="filtered_words",
    outputCol="raw_features",
    vocabSize=2000,
    minDF=20
)

cv_model = vectorizer.fit(df_clean)
df_vectorized = cv_model.transform(df_clean)

idf = IDF(
    inputCol="raw_features",
    outputCol="tfidf_features"
)

idf_model = idf.fit(df_vectorized)
df_tfidf = idf_model.transform(df_vectorized)

In [8]:
vocab_size = len(cv_model.vocabulary)
print("Vocabulary size:", vocab_size)

Vocabulary size: 2000


In [11]:
from pyspark.ml.feature import PCA

pca_k = min(50, vocab_size)

pca = PCA(
    k=pca_k,
    inputCol="tfidf_features",
    outputCol="features"
)

pca_model = pca.fit(df_tfidf)
df_pca = pca_model.transform(df_tfidf)

In [12]:
explained_variance = pca_model.explainedVariance.toArray()

print("Explained variance by first components:")
print(explained_variance[:10])

print("Total explained variance:", explained_variance.sum())

Explained variance by first components:
[0.01373933 0.01244594 0.01103836 0.00839965 0.00733528 0.00671839
 0.00597715 0.00557864 0.00553834 0.00506535]
Total explained variance: 0.21222203919034474


In [13]:
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

evaluator = ClusteringEvaluator(
    featuresCol="features",
    predictionCol="prediction",
    metricName="silhouette"
)

results = []

for k in [4, 6, 8, 10, 12]:
    kmeans = KMeans(
        featuresCol="features",
        predictionCol="prediction",
        k=k,
        seed=42,
        maxIter=20
    )
    
    model = kmeans.fit(df_pca)
    predictions = model.transform(df_pca)
    
    silhouette = evaluator.evaluate(predictions)
    cost = model.summary.trainingCost
    
    results.append((k, silhouette, cost))
    
    print(f"K={k} | Silhouette={silhouette:.4f} | Cost={cost:.2f}")

26/04/25 10:34:52 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS


K=4 | Silhouette=0.3961 | Cost=12021666.14


K=6 | Silhouette=0.2493 | Cost=11282854.92


K=8 | Silhouette=0.2390 | Cost=10337579.45


K=10 | Silhouette=0.1838 | Cost=9623471.71


K=12 | Silhouette=0.4763 | Cost=9653001.25


In [14]:
best_k, best_silhouette, best_cost = max(results, key=lambda x: x[1])

print("Best K:", best_k)
print("Best silhouette:", best_silhouette)
print("Best cost:", best_cost)

Best K: 12
Best silhouette: 0.4762611520023821
Best cost: 9653001.252603227


In [15]:
kmeans = KMeans(
    featuresCol="features",
    predictionCol="cluster_id",
    k=best_k,
    seed=42,
    maxIter=20
)

cluster_model = kmeans.fit(df_pca)
clustered_text = cluster_model.transform(df_pca)

In [16]:
clustered_text.groupBy("cluster_id").count().orderBy("cluster_id").show()

+----------+-----+
|cluster_id|count|
+----------+-----+
|         0| 4157|
|         1|  697|
|         2| 7585|
|         3| 1043|
|         4|   34|
|         5|  348|
|         6|79347|
|         7|  300|
|         8|   83|
|         9| 1064|
|        10| 1273|
|        11|   48|
+----------+-----+



In [17]:
for i in range(best_k):
    print(f"\nCluster {i}")
    (
        clustered_text
        .filter(col("cluster_id") == i)
        .select("cve_id", "description_short")
        .show(5, truncate=False)
    )


Cluster 0
+--------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

+--------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

+--------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [18]:
clustered_ids = clustered_text.select("cve_id", "cluster_id")

full_df = spark.read.parquet("../data/dev/gold/vulnerability_scores")

clustered_df = (
    full_df
    .join(clustered_ids, on="cve_id", how="left")
)

clustered_df.write.mode("overwrite").parquet("../data/dev/gold/vulnerabilities_clustered")

In [19]:
clustered_df.groupBy("cluster_id").agg(
    {"priority_score": "avg"}
).show()

+----------+-------------------+
|cluster_id|avg(priority_score)|
+----------+-------------------+
|         1|0.24281004304160692|
|         6| 0.2320540890014711|
|         3| 0.2572672099712372|
|         5| 0.2322899425287355|
|         9|0.15803468045112762|
|         4| 0.1937705882352941|
|         8|0.26601566265060245|
|         7|0.19774266666666643|
|        10| 0.2532416339355854|
|        11|         0.19933125|
|         2| 0.2603224785761376|
|         0|0.20528316093336516|
+----------+-------------------+



In [17]:
from pyspark.sql.functions import sum

clustered_df.groupBy("cluster_id").agg(
    sum("is_kev").alias("kev_count")
).show()

+----------+---------+
|cluster_id|kev_count|
+----------+---------+
|         1|        4|
|         6|        1|
|         3|        5|
|         5|        0|
|         4|        0|
|         7|        0|
|         2|        0|
|         0|      369|
+----------+---------+

